# Constraint-based discovery (PC / FCI / GFCI / CD-NOD)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abel-ai-lab/andrey/blob/main/examples/constraint_based.ipynb)

The constraint-based methods use conditional-independence tests instead of a score. `pc` returns a
CPDAG; `fci` returns a PAG (partial ancestral graph) that stays sound under latent confounding.
`gfci` builds a PAG from a GES graph, and `cdnod` learns from data pooled from several domains.

## Start

Choose a CPU runtime and run this cell first. In Colab it installs `andrey-core==0.6.0` from PyPI;
local kernels use the installed environment.

In [1]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    os.environ["ANDREY_DEVICE"] = "cpu"
    !pip install --quiet "andrey-core[viz]==0.6.0" pandas

In [2]:
import numpy as np
import pandas as pd

import andrey

The same linear-Gaussian chain `x0 -> x1 -> x2`. Each example here lists its columns in an order
drawn from the seed, so their order says nothing about the causal order; `shuffle=False` keeps the
order the variables are written in.

In [3]:
CHAIN = [("x0", "x1"), ("x1", "x2")]  # the true edges, cause first


def sample_chain(seed: int, n: int = 1500, shuffle: bool = True) -> pd.DataFrame:
    """Draw n rows of the chain, its columns in an order drawn from the seed."""
    rng = np.random.default_rng(seed)
    x0 = rng.standard_normal(n)
    x1 = 0.8 * x0 + rng.standard_normal(n)
    x2 = 0.8 * x1 + rng.standard_normal(n)
    frame = pd.DataFrame({"x0": x0, "x1": x1, "x2": x2})
    return frame[rng.permutation(frame.columns)] if shuffle else frame


X = sample_chain(seed=2)
print("columns:", list(X.columns))

columns: ['x2', 'x1', 'x0']


PC with a Fisher-Z independence test at alpha = 0.05.

In [4]:
pc_out = andrey.pc(X, alpha=0.05)
print("PC  ->", pc_out.structure.kind, pc_out.structure.oriented_edges())

PC  -> cpdag [('x2', 'x1', 'undirected'), ('x1', 'x0', 'undirected')]


FCI returns a PAG, whose endpoint marks encode possible latent confounding.

In [5]:
fci_out = andrey.fci(X, alpha=0.05)
print("FCI ->", fci_out.structure.kind)

FCI -> pag


In [6]:
import andrey.viz

andrey.viz.draw(pc_out.structure)

'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 360 240" width="100%" font-family="Inter, system-ui, sans-serif"><rect width="360" height="240" fill="#fafbfc"/><line x1="93.00" y1="120.00" x2="167.00" y2="120.00" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="193.00" y1="120.00" x2="267.00" y2="120.00" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><circle cx="80.00" cy="120.00" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="80.00" y="120.00" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">x2</text><circle cx="180.00" cy="120.00" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="180.00" y="120.00" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">x1</text><circle cx="280.00" cy="120.00" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="280.00" y="120.00" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">x0</text></svg>'

## GFCI

GFCI starts from the GES graph, removes the edges that independence tests reject, and
orients the rest with FCI's rules, so it also returns a PAG. Here `c` and `d` share a hidden
cause, and `a` and `b` each cause one of them; the PAG marks `c <-> d`.

In [7]:
HIDDEN = [("h", "c"), ("h", "d"), ("a", "c"), ("b", "d")]  # h has no column


def sample_hidden(seed: int, n: int = 2000, shuffle: bool = True) -> pd.DataFrame:
    """Draw n rows of a, b, c, d, whose hidden common cause h is left out."""
    rng = np.random.default_rng(seed)
    hidden = rng.standard_normal(n)
    a = rng.standard_normal(n)
    b = rng.standard_normal(n)
    c = 0.8 * a + 0.8 * hidden + rng.standard_normal(n)
    d = 0.8 * b + 0.8 * hidden + rng.standard_normal(n)
    frame = pd.DataFrame({"a": a, "b": b, "c": c, "d": d})
    return frame[rng.permutation(frame.columns)] if shuffle else frame


gfci_out = andrey.gfci(sample_hidden(seed=3))
print(gfci_out.structure.oriented_edges())
andrey.viz.draw(gfci_out.structure)

[('d', 'c', 'bidirected'), ('b', 'd', 'partially_directed'), ('a', 'c', 'partially_directed')]


'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 360 240" width="100%" font-family="Inter, system-ui, sans-serif"><rect width="360" height="240" fill="#fafbfc"/><line x1="148.76" y1="142.17" x2="211.24" y2="97.83" stroke="#d6495b" stroke-width="2.0" stroke-linecap="round"/><polygon points="140.60,147.96 151.65,146.25 145.86,138.10" fill="#d6495b" stroke-linejoin="round"/><polygon points="219.40,92.04 208.35,93.75 214.14,101.90" fill="#d6495b" stroke-linejoin="round"/><line x1="153.00" y1="155.48" x2="208.60" y2="155.48" stroke="#7c5cd8" stroke-width="2.0" stroke-linecap="round"/><polygon points="143.00,155.48 153.00,160.48 153.00,150.48" fill="#7c5cd8" stroke-linejoin="round"/><circle cx="212.80" cy="155.48" r="4.20" fill="#ffffff" stroke="#7c5cd8" stroke-width="2.0"/><line x1="151.40" y1="84.52" x2="207.00" y2="84.52" stroke="#7c5cd8" stroke-width="2.0" stroke-linecap="round"/><circle cx="147.20" cy="84.52" r="4.20" fill="#ffffff" stroke="#7c5cd8" stroke-width="2.0"/><polygon points="217.00,84.52 207.00,79.52 207.00,89.52" fill="#7c5cd8" stroke-linejoin="round"/><circle cx="130.00" cy="155.48" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="130.00" y="155.48" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">d</text><circle cx="130.00" cy="84.52" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="130.00" y="84.52" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">a</text><circle cx="230.00" cy="84.52" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="230.00" y="84.52" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">c</text><circle cx="230.00" cy="155.48" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="230.00" y="155.48" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">b</text></svg>'

## CD-NOD

CD-NOD takes rows pooled from several domains and a domain index, one value per row. Here the
mean of `x` moves between two domains. PC leaves the chain `x - y - z` undirected; the index
orients it.

In [8]:
DOMAINS = [("x", "y"), ("y", "z")]  # the true edges, cause first


def sample_domains(
    seed: int, n: int = 1000, shuffle: bool = True
) -> tuple[pd.DataFrame, np.ndarray]:
    """Draw n rows from two domains, and the domain of each row; the mean of x moves with it."""
    rng = np.random.default_rng(seed)
    domain = np.repeat([0.0, 1.0], n // 2)  # the first n / 2 rows come from domain 0
    x = rng.standard_normal(n) + 1.5 * domain
    y = 0.8 * x + rng.standard_normal(n)
    z = 0.8 * y + rng.standard_normal(n)
    frame = pd.DataFrame({"x": x, "y": y, "z": z})
    return (frame[rng.permutation(frame.columns)] if shuffle else frame), domain


cdnod_out = andrey.cdnod(*sample_domains(seed=4))
print(cdnod_out.structure.oriented_edges())
andrey.viz.draw(cdnod_out.structure)

[('x', 'y', 'directed'), ('y', 'z', 'directed')]


'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 360 240" width="100%" font-family="Inter, system-ui, sans-serif"><rect width="360" height="240" fill="#fafbfc"/><line x1="157.00" y1="120.00" x2="93.00" y2="120.00" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="167.00,120.00 157.00,115.00 157.00,125.00" fill="#3a4658" stroke-linejoin="round"/><line x1="193.00" y1="120.00" x2="257.00" y2="120.00" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="267.00,120.00 257.00,115.00 257.00,125.00" fill="#3a4658" stroke-linejoin="round"/><circle cx="180.00" cy="120.00" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="180.00" y="120.00" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">y</text><circle cx="80.00" cy="120.00" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="80.00" y="120.00" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">x</text><circle cx="280.00" cy="120.00" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="280.00" y="120.00" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">z</text></svg>'